In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import wrds
db = wrds.Connection(wrds_username = 'xiaoyashi24')

Loading library list...
Done


# Interactive Monthly Stock Performance Analysis Tool

**Module Track:** Interactive Data Analysis Tool  
**Dataset:** CSMAR Monthly Stock Trading Data  
**Table:** `csmar.trd_mnth`  
**Author:** [XiaoyaShi]  
**Date Accessed:** [10 April 2026]

---

## Introduction
This project develops a small Python-based interactive tool for analysing monthly stock performance.
Users can input a stock code and a starting month, and the tool retrieves monthly stock data,
cleans and transforms the data, performs descriptive analysis, and generates visualisations and insights.

## Analytical Problem
How can users quickly evaluate the monthly performance and volatility of a selected stock using Python and financial market data?

## Target Users
The target users are business students, finance learners, and beginner investors.

In [ ]:
# User inputs
stock_code = ''
start_month = 

print("Selected stock code:", stock_code)
print("Selected start month:", start_month) 

In [ ]:
query = f"""
SELECT stkcd, trdmnt, mclsprc, mretwd, mretnd, mnshrtrd, mnvaltrd
FROM csmar.trd_mnth
WHERE stkcd = '{stock_code}'
  AND trdmnt >= {start_month}
"""

df = db.raw_sql(query)

print("Data shape:", df.shape)
df.head()

In [ ]:
if df.empty:
    print("No data found. Please check the stock code or start month.")
else:
    print("Data successfully retrieved.")

In [ ]:
# Keep useful columns
df = df[['stkcd', 'trdmnt', 'mclsprc', 'mretwd', 'mretnd', 'mnshrtrd', 'mnvaltrd']].copy()

# Remove rows with key missing values
df = df.dropna(subset=['trdmnt', 'mretwd', 'mclsprc'])

# Convert trdmnt to datetime
df['trdmnt'] = pd.to_datetime(df['trdmnt'], errors='coerce')

# Drop rows where date conversion failed
df = df.dropna(subset=['trdmnt'])

# Sort by date
df = df.sort_values('trdmnt').reset_index(drop=True)

print("Cleaned data shape:", df.shape)
df.head()

In [ ]:
# Create year-month label
df['year_month'] = df['trdmnt'].dt.strftime('%Y-%m')

# Calculate cumulative return
df['cum_return'] = (1 + df['mretwd']).cumprod() - 1

# Calculate monthly price change
df['price_change'] = df['mclsprc'].diff()

df.head()

In [ ]:
print("Summary statistics for monthly return (mretwd):")
print(df['mretwd'].describe())

print("\nSummary statistics for monthly closing price (mclsprc):")
print(df['mclsprc'].describe())

In [ ]:
mean_return = df['mretwd'].mean()
std_return = df['mretwd'].std()
max_return = df['mretwd'].max()
min_return = df['mretwd'].min()

mean_price = df['mclsprc'].mean()
std_price = df['mclsprc'].std()
max_price = df['mclsprc'].max()
min_price = df['mclsprc'].min()

best_month = df.loc[df['mretwd'].idxmax()]
worst_month = df.loc[df['mretwd'].idxmin()]

print("Average monthly return:", round(mean_return, 4))
print("Return volatility (std):", round(std_return, 4))
print("Highest monthly return:", round(max_return, 4))
print("Lowest monthly return:", round(min_return, 4))

print("\nAverage monthly closing price:", round(mean_price, 2))
print("Price volatility (std):", round(std_price, 2))
print("Highest monthly closing price:", round(max_price, 2))
print("Lowest monthly closing price:", round(min_price, 2))

print("\nBest month:")
print(best_month)

print("\nWorst month:")
print(worst_month)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['trdmnt'], df['mretwd'], marker='o')
plt.title(f'Monthly Return Trend for Stock {stock_code}')
plt.xlabel('Month')
plt.ylabel('Monthly Return')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['trdmnt'], df['cum_return'], marker='o', color='green')
plt.title(f'Cumulative Return for Stock {stock_code}')
plt.xlabel('Month')
plt.ylabel('Cumulative Return')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['trdmnt'], df['mclsprc'], marker='o', color='orange')
plt.title(f'Monthly Closing Price Trend for Stock {stock_code}')
plt.xlabel('Month')
plt.ylabel('Monthly Closing Price')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.bar(df['year_month'], df['mretwd'], color='steelblue')
plt.title(f'Monthly Return Bar Chart for Stock {stock_code}')
plt.xlabel('Month')
plt.ylabel('Monthly Return')
plt.xticks(rotation=90)
plt.show()

In [ ]:
print("Analytical Insight")

if mean_return > 0:
    print(f"- Stock {stock_code} delivered a positive average monthly return over the selected period.")
else:
    print(f"- Stock {stock_code} delivered a negative average monthly return over the selected period.")

if std_return > 0.1:
    print("- The stock shows relatively high volatility in monthly returns.")
else:
    print("- The stock shows relatively moderate monthly return volatility.")

print(f"- The best-performing month was {best_month['trdmnt'].strftime('%Y-%m')}, with a return of {best_month['mretwd']:.4f}.")
print(f"- The worst-performing month was {worst_month['trdmnt'].strftime('%Y-%m')}, with a return of {worst_month['mretwd']:.4f}.")

if df['cum_return'].iloc[-1] > 0:
    print("- Overall cumulative return is positive over the selected period.")
else:
    print("- Overall cumulative return is negative over the selected period.")

if df['mclsprc'].iloc[-1] > df['mclsprc'].iloc[0]:
    print("- The monthly closing price shows an overall upward trend.")
else:
    print("- The monthly closing price does not show a clear upward trend.")

In [ ]:
import os

output_file = f"{stock_code}_monthly_stock_analysis.xlsx"
df.to_excel(output_file, index=False)

print("File saved as:", output_file)
print("Saved in folder:", os.getcwd())

## Methods Used

This project applies the following Python-based analytical steps:

1. **Data acquisition**  
   Monthly stock data were retrieved from the CSMAR `trd_mnth` table.

2. **Data cleaning**  
   Missing values were removed and only relevant variables were kept.

3. **Data transformation**  
   The month variable was converted to datetime format, cumulative return was calculated,
   and monthly price changes were derived.

4. **Descriptive analysis**  
   Key statistics such as average return, volatility, best month, and worst month were calculated.

5. **Visualisation**  
   Line charts and bar charts were used to present stock performance patterns clearly.

## Conclusion

This project developed a small interactive Python-based stock analysis tool using a business-related dataset from CSMAR.
The tool allows users to input a stock code and a starting month to retrieve monthly stock data and perform
data cleaning, transformation, descriptive analysis, and visualisation.

The final output communicates analytical value by helping users understand stock return behaviour, price trends,
and volatility over time. This makes the tool suitable for business students, finance learners, and beginner investors.